In [1]:
import csv
import datetime
import hashlib
import json
import os
import time

import requests


try:
    from kaggle_secrets import UserSecretsClient

    secret_client = UserSecretsClient()
    try:
        os.environ["LLM_API_KEY"] = secret_client.get_secret("DOJ")
    except Exception:
        os.environ["LLM_API_KEY"] = secret_client.get_secret("LLM_API_KEY")
except Exception as exc:
    print("No Kaggle secret found, falling back to the environment variable:", exc)

API_KEY = os.environ.get("LLM_API_KEY", "")

DATE_OF_USE = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
CFG = {
    "api_base": "https://api.groq.com/openai/v1",
    "model": "openai/gpt-oss-20b",
    "temperature": 0.0,
    "max_tokens": 4096,
    "seed": 42,
    "json_mode": True,
    "reasoning_effort": "low",
}
HEAD = {
    "Authorization": f"Bearer {API_KEY}",
    "User-Agent": "ITFFC-student-workshop/1.0",
}

FRAUD_CSV = "/kaggle/input/datasets/abdelilehlouhichi/fraud-final/fraud_selected_final.csv"
FACTS_CSV = "/kaggle/input/datasets/abdelilehlouhichi/facts-final/facts_sheet_final.csv"
OUTPUT_DIR = "/kaggle/working"

if not os.path.isdir("/kaggle"):
    FRAUD_CSV = "result/fraud_selected_final.csv"
    FACTS_CSV = "result/facts_sheet_final.csv"
    OUTPUT_DIR = "result"

os.makedirs(OUTPUT_DIR, exist_ok=True)

KEY_FIELDS = (
    "defendant_name",
    "total_amount",
    "case_stage",
    "sentence",
    "date",
)
VALID_CASE_STAGES = {"charged", "pleaded guilty", "convicted", "sentenced"}

COMMON_OUTPUT_RULES = """
Return exactly one valid JSON object with these five keys:
- "defendant_name": the defendant's name, or "not stated"
- "total_amount": the amount stated in the source, or "not stated"
- "case_stage": exactly one of "charged", "pleaded guilty", "convicted", "sentenced", or "not stated"
- "sentence": the sentence or punishment stated in the source, or "not stated"
- "date": the relevant date stated in the source, or "not stated"

Any fact absent from the provided source must be written exactly as "not stated".
Do not add fields outside the five-key object.
"""

BASELINE_PROMPT = (
    "Extract the five report fields from the release title below.\n"
    + COMMON_OUTPUT_RULES
)

GROUNDED_PROMPT = (
    "Extract the five report fields using only the release title and cleaned text below.\n"
    "Use only evidence in the provided text. Do not use outside knowledge.\n"
    "Do not present an allegation as a conviction. If the stage is not clearly supported, "
    "use \"not stated\".\n"
    + COMMON_OUTPUT_RULES
)


def read_csv(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def prompt_hash(prompt):
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()


def inspect_calibration_data():
    """Uses fraud cases for prompt review only; it does not train or fine-tune the model."""
    rows = read_csv(FRAUD_CSV)
    calibration_manifest = {
        "source": FRAUD_CSV,
        "row_count": len(rows),
        "purpose": "Manual prompt review; no model training or fine-tuning",
        "baseline_prompt_sha256": prompt_hash(BASELINE_PROMPT),
        "grounded_prompt_sha256": prompt_hash(GROUNDED_PROMPT),
        "sample_uuids": [row.get("uuid", "") for row in rows[:30]],
    }
    manifest_path = os.path.join(OUTPUT_DIR, "fraud_calibration_manifest.json")
    with open(manifest_path, "w", encoding="utf-8") as handle:
        json.dump(calibration_manifest, handle, ensure_ascii=False, indent=2)
    print(
        "Fraud calibration rows:",
        len(rows),
        "| no API calls made; manifest:",
        manifest_path,
    )


def validate_api():
    if not API_KEY:
        raise RuntimeError(
            'Missing Kaggle secret "DOJ" or "LLM_API_KEY". '
            "Add it as a secret in Kaggle Notebook settings."
        )

    print(
        "key length:",
        len(API_KEY),
        "| date of use:",
        DATE_OF_USE,
        "| model:",
        CFG["model"],
    )
    response = requests.get(
        f"{CFG['api_base']}/models",
        headers=HEAD,
        timeout=60,
    )
    response.raise_for_status()
    ids = [item["id"] for item in response.json().get("data", [])]
    print("Groq models available:", ids[:20])
    if CFG["model"] not in ids:
        print(
            f"WARNING: CFG['model'] ({CFG['model']}) was not returned "
            "by the available-models endpoint."
        )

    smoke_payload = {
        "model": CFG["model"],
        "temperature": 0,
        "max_tokens": 200,
        "messages": [{"role": "user", "content": 'Reply with the JSON {"ok": true}'}],
    }
    if CFG["json_mode"]:
        smoke_payload["response_format"] = {"type": "json_object"}
    smoke_response = requests.post(
        f"{CFG['api_base']}/chat/completions",
        json=smoke_payload,
        headers=HEAD,
        timeout=120,
    )
    print("Groq smoke test:", smoke_response.status_code, smoke_response.text[:400])
    smoke_response.raise_for_status()


def strip_code_fence(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines:
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return text


def parse_report(raw_answer):
    text = strip_code_fence(raw_answer)
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end > start:
            try:
                parsed = json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                parsed = {}
        else:
            parsed = {}

    if not isinstance(parsed, dict):
        return {}, False

    report = {}
    for field in KEY_FIELDS:
        value = parsed.get(field, "not stated")
        report[field] = value if isinstance(value, str) and value.strip() else "not stated"

    stage = report["case_stage"].strip()
    if stage not in VALID_CASE_STAGES and stage != "not stated":
        report["case_stage"] = "not stated"

    valid = set(report) == set(KEY_FIELDS)
    valid = valid and set(parsed) == set(KEY_FIELDS)
    valid = valid and all(isinstance(report[field], str) for field in KEY_FIELDS)
    return report, valid


def build_payload(prompt, title, text=None):
    if text is None:
        input_payload = f"Title:\n{title}"
    else:
        input_payload = f"Title:\n{title}\n\nCleaned text:\n{text}"

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": input_payload},
    ]
    payload = {
        "model": CFG["model"],
        "temperature": CFG["temperature"],
        "max_tokens": CFG["max_tokens"],
        "messages": messages,
        "seed": CFG["seed"],
    }
    if CFG["reasoning_effort"]:
        payload["reasoning_effort"] = CFG["reasoning_effort"]
    if CFG["json_mode"]:
        payload["response_format"] = {"type": "json_object"}
    return payload, input_payload


def generate_once(payload):
    response = requests.post(
        f"{CFG['api_base']}/chat/completions",
        json=payload,
        headers=HEAD,
        timeout=120,
    )
    if response.status_code >= 400:
        raise requests.HTTPError(
            f"{response.status_code}: {response.text[:500]}",
            response=response,
        )
    result = response.json()
    choices = result.get("choices", [])
    if not choices:
        raise RuntimeError(f"No choices in API response: {json.dumps(result)[:500]}")
    return choices[0]["message"]["content"]


def generate_with_retries(payload, attempts=4):
    for attempt in range(1, attempts + 1):
        try:
            return generate_once(payload)
        except Exception as exc:
            if attempt == attempts:
                raise
            delay = 2 ** attempt
            print(
                f"Retry {attempt}/{attempts - 1} after {type(exc).__name__}; "
                f"waiting {delay}s"
            )
            time.sleep(delay)


def load_checkpoint(path):
    rows = {}
    if not os.path.exists(path):
        return rows
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            rows[(row["uuid"], row["generator"])] = row
    return rows


def write_checkpoint(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for row in sorted(rows.values(), key=lambda item: (item["uuid"], item["generator"])):
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path, rows):
    fieldnames = [
        "uuid",
        "generator",
        "input",
        "raw_answer",
        "parsed_report",
        "validity",
        "model",
        "date_of_use",
        "temperature",
        "max_tokens",
        "seed",
        "reasoning_effort",
        "prompt_sha256",
    ]
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in sorted(rows.values(), key=lambda item: (item["uuid"], item["generator"])):
            writer.writerow(row)


def main():
    inspect_calibration_data()
    validate_api()

    test_rows = read_csv(FACTS_CSV)
    test_limit = int(os.environ.get("TEST_LIMIT", "0") or 0)
    if test_limit > 0:
        test_rows = test_rows[:test_limit]
    print("Facts-sheet test rows:", len(test_rows))

    jsonl_path = os.path.join(OUTPUT_DIR, "part2_reports_checkpoint.jsonl")
    csv_path = os.path.join(OUTPUT_DIR, "part2_generated_reports.csv")
    rows = load_checkpoint(jsonl_path)
    completed_before = len(rows)

    for index, case in enumerate(test_rows, start=1):
        uuid = case.get("uuid", "").strip()
        title = case.get("title", "").strip()
        cleaned_text = case.get("body_text", "").strip()
        if not uuid:
            raise ValueError(f"Facts-sheet row {index} has no uuid")

        for generator, prompt, include_text in (
            ("baseline", BASELINE_PROMPT, False),
            ("grounded", GROUNDED_PROMPT, True),
        ):
            key = (uuid, generator)
            if key in rows:
                continue

            payload, input_payload = build_payload(
                prompt,
                title,
                cleaned_text if include_text else None,
            )
            try:
                raw_answer = generate_with_retries(payload)
                parsed_report, validity = parse_report(raw_answer)
            except Exception as exc:
                raw_answer = ""
                parsed_report = {}
                validity = False
                print(
                    f"Error for uuid={uuid} generator={generator}: "
                    f"{type(exc).__name__}: {exc}"
                )

            rows[key] = {
                "uuid": uuid,
                "generator": generator,
                "input": input_payload,
                "raw_answer": raw_answer,
                "parsed_report": json.dumps(parsed_report, ensure_ascii=False),
                "validity": bool(validity),
                "model": CFG["model"],
                "date_of_use": DATE_OF_USE,
                "temperature": CFG["temperature"],
                "max_tokens": CFG["max_tokens"],
                "seed": CFG["seed"],
                "reasoning_effort": CFG["reasoning_effort"],
                "prompt_sha256": prompt_hash(prompt),
            }
            write_checkpoint(jsonl_path, rows)
            write_csv(csv_path, rows)
            print(
                f"Saved {index}/{len(test_rows)} {generator} "
                f"(valid={validity}, uuid={uuid})"
            )

        time.sleep(0.25)

    print(
        "Completed.",
        "before this run:",
        completed_before,
        "| checkpoint rows:",
        len(rows),
        "| CSV:",
        csv_path,
    )


if __name__ == "__main__":
    main()


# ---- Kaggle presentation outputs (append-only evaluation and tables) ----

def build_evaluation_outputs():
    """Create an image-like metric table and detailed HTML/CSV tables."""
    try:
        import pandas as pd
        from IPython.display import HTML, display
    except ImportError as exc:
        print("Evaluation display skipped:", exc)
        return

    generated_path = os.path.join(OUTPUT_DIR, "part2_generated_reports.csv")
    if not os.path.exists(generated_path):
        print("Generated report file not found:", generated_path)
        return

    facts = pd.read_csv(FACTS_CSV)
    generated = pd.read_csv(generated_path)
    generated["parsed_report"] = generated["parsed_report"].fillna("{}")

    def safe_json(value):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except (json.JSONDecodeError, TypeError):
            return {}

    generated["report"] = generated["parsed_report"].map(safe_json)
    for field in KEY_FIELDS:
        generated[field] = generated["report"].map(
            lambda report, field=field: (
                report.get(field).strip()
                if isinstance(report.get(field), str) and report.get(field).strip()
                else "not stated"
            )
        )

    merged = generated.merge(
        facts,
        on="uuid",
        how="left",
        suffixes=("_generated", "_source"),
    )

    def text_supported(row, field):
        value = str(row[field]).strip()
        if value.lower() == "not stated":
            return False
        body = str(row.get("body_text", "")).lower()
        return value.lower() in body

    def amount_supported(row):
        value = str(row["total_amount"]).strip()
        if value.lower() == "not stated":
            return False
        amounts = str(row.get("amounts_found", "")).lower()
        body = str(row.get("body_text", "")).lower()
        candidate = value.lower().replace(" ", "").replace(",", "")
        return (
            candidate in amounts.replace(" ", "").replace(",", "")
            or candidate in body.replace(" ", "").replace(",", "")
        )

    merged["amount_supported"] = merged.apply(amount_supported, axis=1)
    merged["names_supported"] = merged.apply(
        lambda row: text_supported(row, "defendant_name"),
        axis=1,
    )

    for field in KEY_FIELDS:
        merged[f"ns_{field}"] = (
            merged[field].fillna("").str.strip().str.lower() == "not stated"
        )

    summary = (
        merged.groupby("generator")
        .agg(
            valid_attempts=("validity", "sum"),
            n=("uuid", "count"),
            amount_supported=("amount_supported", "mean"),
            names_supported=("names_supported", "mean"),
        )
        .reset_index()
    )

    for field in KEY_FIELDS:
        summary[f"ns_{field}"] = (
            merged.groupby("generator")[f"ns_{field}"].mean().reindex(summary["generator"]).values
        )

    summary = summary[
        ["generator", "valid_attempts", "n", "amount_supported", "names_supported"]
        + [f"ns_{field}" for field in KEY_FIELDS]
    ]
    summary["valid_attempts"] = summary["valid_attempts"].round().astype(int)
    for column in ("amount_supported", "names_supported") + tuple(
        f"ns_{field}" for field in KEY_FIELDS
    ):
        summary[column] = (100 * summary[column]).round(2).astype(str) + "%"

    print("Your connection is stuck" if False else "Part 2 - metric summary")
    display(
        HTML(
            summary.to_html(
                index=False,
                escape=True,
                classes="table table-striped",
            )
        )
    )

    stage_table = pd.crosstab(
        merged["generator"],
        merged["case_stage"],
        margins=True,
        margins_name="total",
    )
    print("Case-stage counts")
    display(HTML(stage_table.to_html(classes="table table-bordered")))

    detail = merged[
        [
            "uuid",
            "generator",
            "title",
            "url",
            "defendant_name",
            "total_amount",
            "case_stage",
            "sentence",
            "date",
            "validity",
            "amount_supported",
            "names_supported",
            "raw_answer",
        ]
    ].copy()
    detail["title_link"] = detail.apply(
        lambda row: (
            f'<a href="{row["url"]}" target="_blank">'
            f'{str(row["title"]).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")}'
            "</a>"
            if pd.notna(row["url"])
            else row["title"]
        ),
        axis=1,
    )
    detail = detail[
        [
            "uuid",
            "generator",
            "title_link",
            "defendant_name",
            "total_amount",
            "case_stage",
            "sentence",
            "date",
            "validity",
            "amount_supported",
            "names_supported",
            "raw_answer",
        ]
    ].rename(columns={"title_link": "release", "validity": "valid"})

    detail_path = os.path.join(OUTPUT_DIR, "part2_results_with_links.csv")
    detail.to_csv(detail_path, index=False)
    print("Detailed linked table saved at:", detail_path)
    display(
        HTML(
            detail.to_html(
                index=False,
                escape=False,
                classes="table table-hover",
                max_rows=100,
            )
        )
    )


if __name__ == "__main__":
    main()
    build_evaluation_outputs()

Fraud calibration rows: 2878 | no API calls made; manifest: /kaggle/working/fraud_calibration_manifest.json
key length: 56 | date of use: 2026-11-20 | model: openai/gpt-oss-20b
Groq models available: ['openai/gpt-oss-20b', 'whisper-large-v3-turbo', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-safeguard-20b', 'whisper-large-v3', 'canopylabs/orpheus-arabic-saudi', 'allam-2-7b', 'meta-llama/llama-prompt-guard-2-22m', 'qwen/qwen3.8-27b', 'canopylabs/orpheus-v1-english']
Groq smoke test: 200 {"id":"chatcmpl-f30d9a56-fc81-4266-a1da-0862f4a8ad0a","object":"chat.completion","created":1795189410,"model":"openai/gpt-oss-20b","choices":[{"index":0,"message":{"role":"assistant","content":"{\"ok\":true}","reasoning":"The user wants a JSON {\"ok\": true}. We need to output that JSON. Ensure it's compact. So output: {\"ok\":true}."},"logprobs":null,"finish_reason":"stop"}],"usage":{"queue_time"
Facts-sheet test rows: 30
Saved 1/30 baseline (valid=True, uuid=abc9c8fc-f

generator,valid_attempts,n,amount_supported,names_supported,ns_defendant_name,ns_total_amount,ns_case_stage,ns_sentence,ns_date
baseline,30,30,3.33%,10.0%,70.0%,93.33%,10.0%,66.67%,100.0%
grounded,30,30,73.33%,100.0%,0.0%,20.0%,6.67%,53.33%,40.0%


Case-stage counts


case_stage,charged,convicted,not stated,pleaded guilty,sentenced,total
generator,,,,,,
baseline,5,0,3,10,12,30
grounded,5,2,2,11,10,30
total,10,2,5,21,22,60


Detailed linked table saved at: /kaggle/working/part2_results_with_links.csv


uuid,generator,release,defendant_name,total_amount,case_stage,sentence,date,valid,amount_supported,names_supported,raw_answer
14fadf73-9c2b-4b50-9c5b-5b25e2fe0e6b,baseline,Former Employee of National Industrial Company Pleads Guilty to Crimes Related to Hacking Computer Networks and Extorting Employees,not stated,not stated,pleaded guilty,not stated,not stated,True,False,False,"{""defendant_name"":""not stated"",""total_amount"":""not stated"",""case_stage"":""pleaded guilty"",""sentence"":""not stated"",""date"":""not stated""}"
14fadf73-9c2b-4b50-9c5b-5b25e2fe0e6b,grounded,Former Employee of National Industrial Company Pleads Guilty to Crimes Related to Hacking Computer Networks and Extorting Employees,Daniel Rhyne,"$750,000",pleaded guilty,not stated,"April 1, 2026",True,True,True,"{""defendant_name"":""Daniel Rhyne"",""total_amount"":""$750,000"",""case_stage"":""pleaded guilty"",""sentence"":""not stated"",""date"":""April 1, 2026""}"
1c77bb98-e149-412e-a99c-b31f5e19e42a,baseline,California Man Pleads Guilty to Receiving Child Pornography and Transferring Obscene Material to a Minor,California Man,not stated,pleaded guilty,not stated,not stated,True,False,False,"{""defendant_name"":""California Man"",""total_amount"":""not stated"",""case_stage"":""pleaded guilty"",""sentence"":""not stated"",""date"":""not stated""}"
1c77bb98-e149-412e-a99c-b31f5e19e42a,grounded,California Man Pleads Guilty to Receiving Child Pornography and Transferring Obscene Material to a Minor,Devin Ravine,"$250,000",pleaded guilty,"at least 5 years and up to 30 years in prison, a fine of up to $250,000, a supervised release term of at least 5 years and up to life, forfeiture, and restitution","September 4, 2026",True,True,True,"{""defendant_name"":""Devin Ravine"",""total_amount"":""$250,000"",""case_stage"":""pleaded guilty"",""sentence"":""at least 5 years and up to 30 years in prison, a fine of up to $250,000, a supervised release term of at least 5 years and up to life, forfeiture, and restitution"",""date"":""September 4, 2026""}"
219ff1b4-1d9d-4312-8188-3960a9f5266a,baseline,Depew man pleads guilty to entering secure area at the Buffalo Niagara International Airport,not stated,not stated,pleaded guilty,not stated,not stated,True,False,False,"{""defendant_name"":""not stated"",""total_amount"":""not stated"",""case_stage"":""pleaded guilty"",""sentence"":""not stated"",""date"":""not stated""}"
219ff1b4-1d9d-4312-8188-3960a9f5266a,grounded,Depew man pleads guilty to entering secure area at the Buffalo Niagara International Airport,James R. Muench Jr.,"$100,000",pleaded guilty,not stated,"May 20, 2025",True,True,True,"{""defendant_name"":""James R. Muench Jr."",""total_amount"":""$100,000"",""case_stage"":""pleaded guilty"",""sentence"":""not stated"",""date"":""May 20, 2025""}"
32980bf6-ac61-4d5d-be80-acd6dc6e1fba,baseline,Four Hudson County Pharmacies Pay $4.3 Million To Resolve False Claims Act Allegations of Billing for Drugs Not Dispensed,Four Hudson County Pharmacies,$4.3 Million,not stated,not stated,not stated,True,False,False,"{""defendant_name"":""Four Hudson County Pharmacies"",""total_amount"":""$4.3 Million"",""case_stage"":""not stated"",""sentence"":""not stated"",""date"":""not stated""}"
32980bf6-ac61-4d5d-be80-acd6dc6e1fba,grounded,Four Hudson County Pharmacies Pay $4.3 Million To Resolve False Claims Act Allegations of Billing for Drugs Not Dispensed,Pan American Inc.,"$3,100,000",not stated,not stated,not stated,True,True,True,"{""defendant_name"":""Pan American Inc."",""total_amount"":""$3,100,000"",""case_stage"":""not stated"",""sentence"":""not stated"",""date"":""not stated""}"
3a9e1c24-bd86-4d73-9e2c-711368eede73,baseline,"Two Home Health Providers Agree to Pay $702,541 to Resolve Self-Referral Allegations",not stated,"$702,541",not stated,not stated,not stated,True,True,False,"{""defendant_name"":""not stated"",""total_amount"":""$702,541"",""case_stage"":""not stated"",""sentence"":""not stated"",""date"":""not state